In [ ]:
from ultralytics import YOLO
from ultralytics.utils.downloads import download
import torch
import os
from pathlib import Path
from collections import Counter
import yaml
import random
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

In [ ]:
%matplotlib inline

In [ ]:
#Изначально в датасете аннотации не в YOLO-формате, поэтому скачаем данные и переводим в YOLO-формат с помощью указанного в yaml-файле кода.
with open("VisDrone.yaml", "r") as f:
    data_desc = yaml.safe_load(f)

def visdrone2yolo(dir):
  from PIL import Image
  from tqdm import tqdm

  def convert_box(size, box):
      # Convert VisDrone box to YOLO xywh box
      dw = 1. / size[0]
      dh = 1. / size[1]
      return (box[0] + box[2] / 2) * dw, (box[1] + box[3] / 2) * dh, box[2] * dw, box[3] * dh

  (dir / 'labels').mkdir(parents=True, exist_ok=True)  # make labels directory
  pbar = tqdm((dir / 'annotations').glob('*.txt'), desc=f'Converting {dir}')
  for f in pbar:
      img_size = Image.open((dir / 'images' / f.name).with_suffix('.jpg')).size
      lines = []
      with open(f, 'r') as file:  # read annotation.txt
          for row in [x.split(',') for x in file.read().strip().splitlines()]:
              if row[4] == '0':  # VisDrone 'ignored regions' class 0
                  continue
              cls = int(row[5]) - 1
              box = convert_box(img_size, tuple(map(int, row[:4])))
              lines.append(f"{cls} {' '.join(f'{x:.6f}' for x in box)}\n")
              with open(str(f).replace(f'{os.sep}annotations{os.sep}', f'{os.sep}labels{os.sep}'), 'w') as fl:
                  fl.writelines(lines)  # write label.txt


# Download
dir = Path(data_desc['path'])  # dataset root dir
urls = ['https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-train.zip',
      'https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-val.zip',
      'https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-test-dev.zip',
      'https://github.com/ultralytics/yolov5/releases/download/v1.0/VisDrone2019-DET-test-challenge.zip']
download(urls, dir=dir, curl=True, threads=4)

# Convert
for d in 'VisDrone2019-DET-train', 'VisDrone2019-DET-val', 'VisDrone2019-DET-test-dev':
  visdrone2yolo(dir / d)  # convert VisDrone annotations to YOLO labels


In [ ]:
#проверяем, работает ли GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## YOLOv8n
Обучаю предобученную YOLOv8n с параметрами по умолчанию.

In [ ]:
model_yolov8n_default = YOLO("yolov8n.pt").to(device)
results_yolov8n_default = model_yolov8n_default.train(
    data = "VisDrone.yaml", 
    epochs = 50, 
    batch = 8, 
    device = 0, 
    patience = 20
)

In [ ]:
#валидация
metrics_train = model_yolov8n_default.val(data = "VisDrone.yaml", split = "train", imgsz = 1024)
metrics_val = model_yolov8n_default.val(data = "VisDrone.yaml", imgsz = 1024)

In [ ]:
print("yolov8n_default")
print(f"Train: {metrics_train.box.map}")
print(f"Val: {metrics_val.box.map}")

### Анализ датасета 
Проанализируем особенности данных, чтобы выдвинуть гипотезы, как можно было бы улучшить качество.



#### Частота классов

Некоторые классы определяются значительно хуже, чем другие, возможно, дело в том, что одни классы встречаются значительно чаще. Проверим, есть ли зависимость между частотой класса в датасете и качеством на этом классе.

In [ ]:
def count_classes(path):
    cnt = Counter()
    for file_name in os.listdir(path):
        file_path = os.path.join(path, file_name)
        with open(file_path, "r") as f:
            for line in f:
                nums = line.split()
                class_name = data_desc["names"][int(nums[0])]
                cnt[class_name] += 1
    return cnt

path = "datasets/VisDrone/VisDrone2019-DET-train/labels"
cnt_classes = count_classes(path)
for class_name, num in cnt_classes.most_common():
    print(f"{class_name}: {num}")

In [ ]:
for class_id, class_name in sorted(data_desc["names"].items(), key = lambda x: metrics_val.box.maps[x[0]], reverse = True):
    print(f"{class_name}: {metrics_val.box.maps[class_id]}")

##### Выводы
Нет явной зависимости между метрикой и частотой класса: некоторые частые классы (pedestrian, bicycle) определяются сравнительно плохо, в то время как достаточно редкий bus определяется хорошо. 
Можно сделать предположение, что дело не в частоте, а в размере некоторых классов, таких как люди и велосипеды.

#### Отрисовка изображений
Чем отличаются классы pedestrian и people?

In [ ]:
def load_yolo_bboxes(label_path):
    bboxes = []

    if not label_path.exists():
        return bboxes

    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id, x_center, y_center, width, height = parts
            bboxes.append(
                {
                    "class_id": int(class_id),
                    "x_center": float(x_center),
                    "y_center": float(y_center),
                    "width": float(width),
                    "height": float(height),
                }
            )

    return bboxes


def yolo_to_xyxy(bbox, img_w, img_h):
    x_center = bbox["x_center"] * img_w
    y_center = bbox["y_center"] * img_h
    width = bbox["width"] * img_w
    height = bbox["height"] * img_h

    x1 = int(x_center - width / 2)
    y1 = int(y_center - height / 2)
    x2 = int(x_center + width / 2)
    y2 = int(y_center + height / 2)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(img_w - 1, x2)
    y2 = min(img_h - 1, y2)

    return x1, y1, x2, y2


def find_images_with_class(images_dir, labels_dir, target_class, exts=(".jpg", ".jpeg", ".png", ".bmp")):
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    matched_images = []

    for img_path in images_dir.rglob("*"):
        if img_path.suffix.lower() not in exts:
            continue

        label_path = labels_dir / (img_path.stem + ".txt")
        bboxes = load_yolo_bboxes(label_path)

        if any(b["class_id"] == target_class for b in bboxes):
            matched_images.append(img_path)

    return matched_images


def draw_bboxes(
    image,
    bboxes,
    show_only_class=None,
    color=(0, 255, 0),
    thickness=2,
    draw_label=True,
):
    img_h, img_w = image.shape[:2]
    result = image.copy()

    for bbox in bboxes:
        class_id = bbox["class_id"]

        if show_only_class is not None and class_id != show_only_class:
            continue

        x1, y1, x2, y2 = yolo_to_xyxy(bbox, img_w, img_h)
        cv2.rectangle(result, (x1, y1), (x2, y2), color, thickness)

        if draw_label:
            cv2.putText(
                result,
                f"class {class_id}",
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2,
                cv2.LINE_AA,
            )

    return result


def show_random_samples_by_class(
    images_dir,
    labels_dir,
    target_class,
    num_samples=5,
    show_only_target_class=True,
    figsize=(15, 10),
):
    matched_images = find_images_with_class(images_dir, labels_dir, target_class)

    if not matched_images:
        print(f"Изображения с классом {target_class} не найдены.")
        return

    num_samples = min(num_samples, len(matched_images))
    selected_images = random.sample(matched_images, num_samples)

    cols = min(3, num_samples)
    rows = (num_samples + cols - 1) // cols

    plt.figure(figsize=figsize)

    for i, img_path in enumerate(selected_images, start=1):
        label_path = Path(labels_dir) / (img_path.stem + ".txt")
        bboxes = load_yolo_bboxes(label_path)

        image = cv2.imread(str(img_path))
        if image is None:
            print(f"Не удалось прочитать изображение: {img_path}")
            continue

        drawn = draw_bboxes(
            image=image,
            bboxes=bboxes,
            show_only_class=target_class if show_only_target_class else None,
        )

        drawn_rgb = cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB)

        plt.subplot(rows, cols, i)
        plt.imshow(drawn_rgb)
        plt.title(img_path.name)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
#выводим pedestrian
images_dir = "datasets/VisDrone/VisDrone2019-DET-train/images"
labels_dir = "datasets/VisDrone/VisDrone2019-DET-train/labels"

target_class = 0
num_samples = 6

show_random_samples_by_class(
    images_dir=images_dir,
    labels_dir=labels_dir,
    target_class=target_class,
    num_samples=num_samples,
    show_only_target_class=True,
)

In [ ]:
#выводим people
target_class = 1
show_random_samples_by_class(
    images_dir=images_dir,
    labels_dir=labels_dir,
    target_class=target_class,
    num_samples=num_samples,
    show_only_target_class=True,
)

##### Выводы
pedestrian - это человек, который идет пешком, а people - это человек, который едет на транспорте.
Кроме того, по данным изображениям, действительно видно, что объекты очень мелкие.

## YOLOv8n с размером изображений 1024x1024
Объекты маленькие, так что попробуем увеличить размер изображения.

In [ ]:
model_yolov8n_img1024 = YOLO("yolov8n.pt").to(device)
results_yolov8n_img1024 = model_yolov8n_img1024.train(
    data = "VisDrone.yaml", 
    epochs = 50, 
    batch = 8, 
    device = 0, 
    patience = 10, 
    imgsz = 1024
)

In [ ]:
metrics_train = model_yolov8n_img1024.val(data = "VisDrone.yaml", split = "train", imgsz = 1024)
metrics_val = model_yolov8n_img1024.val(data = "VisDrone.yaml", imgsz = 1024)

In [ ]:
print("yolov8n_img1024")
print(f"Train: {metrics_train.box.map}")
print(f"Val: {metrics_val.box.map}")

Увеличение входного изображения действительно улучшило метрику.

## YOLOv8n с размером изображений 1024x1024 с выключенной mosaic аугментацией
Если объекты слишком маленькие, mosaic аугментация может делать их неразличимыми и снижать качество. Попробуем ее отключить.

In [ ]:
model_yolov8n_img1024_mos = YOLO("yolov8n.pt").to(device)
results_yolov8n_img1024_mos = model_yolov8n_img1024_mos.train(
    data = "VisDrone.yaml", 
    epochs = 50, 
    batch = 8, 
    device = 0, 
    patience = 10, 
    imgsz = 1024, 
    workers = 0, 
    mosaic = 0
)

In [ ]:
metrics_train = model_yolov8n_img1024_mos.val(data = "VisDrone.yaml", split = "train", imgsz = 1024)
metrics_val = model_yolov8n_img1024_mos.val(data = "VisDrone.yaml", imgsz = 1024)

In [ ]:
print("yolov8n_img1024_mos")
print(f"Train: {metrics_train.box.map}")
print(f"Val: {metrics_val.box.map}")

Качество понизилось по сравнению с той же моделью и mosaic = 1

## YOLOv8s с размером изображений 1024x1024
Попробуем большую модель, так как сильного переобучения в прошлый раз не было, возможно, YOLOv8n слишком маленькая для этих данных.

In [ ]:
model_yolov8s_img1024 = YOLO("yolov8s.pt").to(device)
results_yolov8s_img1024 = model_yolov8s_img1024.train(
    data = "VisDrone.yaml", 
    epochs = 100, batch = 8, 
    device = 0, 
    patience = 10, 
    imgsz = 1024, 
    workers = 0
)

In [ ]:
metrics_train = model_yolov8s_img1024.val(data = "VisDrone.yaml", split = "train", imgsz = 1024)
metrics_val = model_yolov8s_img1024.val(data = "VisDrone.yaml", imgsz = 1024)

In [ ]:
print("yolov8s_img1024")
print(f"Train: {metrics_train.box.map}")
print(f"Val: {metrics_val.box.map}")

Качество возросло, действительно, большая модель лучше улавливает закономерности.

## YOLOv8s с размером изображений 1024x1024 c уменьшенными аугментациями
Слишком сильные аугментации могут делать маленькие объекты неразлечимыми, попробуем их ослабить.

In [ ]:
model_yolov8s_img1024_aug = YOLO("yolov8s.pt").to(device)
results_yolov8s_img1024_aug = model_yolov8s_img1024_aug.train(
    data = "VisDrone.yaml", 
    epochs = 100, batch = 8, 
    device = 0, 
    patience = 10, 
    imgsz = 1024, 
    workers = 0,
    scale = 0.3,
    translate = 0.05,
    degrees = 0.0,
    shear = 0.0,
    perspective = 0.0
)

In [ ]:
metrics_train = model_yolov8s_img1024_aug.val(data = "VisDrone.yaml", split = "train", imgsz = 1024)
metrics_val = model_yolov8s_img1024_aug.val(data = "VisDrone.yaml", imgsz = 1024)

In [ ]:
print("yolov8s_img1024_aug")
print(f"Train: {metrics_train.box.map}")
print(f"Val: {metrics_val.box.map}")

Качество упало на валидации и улучшилось на тесте, в данном случае аугментации действительно предотвращали переобучение.

## YOLOv8s с размером изображений 1024x1024 c mosaic = 0.5
Попробуем не так сильно уменьшать mosaic, поставим ему значение 0.5.

In [ ]:
model_yolov8s_img1024_mos = YOLO("yolov8s.pt").to(device)
results_yolov8s_img1024_mos = model_yolov8s_img1024_mos.train(
    data = "VisDrone.yaml", 
    epochs = 100, batch = 8, 
    device = 0, 
    patience = 10, 
    imgsz = 1024, 
    workers = 0,
    mosaic = 0.5
)

In [ ]:
metrics_train = model_yolov8s_img1024_mos.val(data = "VisDrone.yaml", split = "train", imgsz = 1024)
metrics_val = model_yolov8s_img1024_mos.val(data = "VisDrone.yaml", imgsz = 1024)

In [ ]:
print("yolov8s_img1024_mos")
print(f"Train: {metrics_train.box.map}")
print(f"Val: {metrics_val.box.map}")

## Лучшая модель
Лучшее качество показала модель YOLOv8s с размером изображений 1024x1024 и аугментациями по умолчанию. 

Увеличение размера изображение и большая модель помогли при работе с мелкими объектами, а аугментации не ухудшали качество, а увеличивали обобщающую способность модели. 

### Финальная валидация на тесте лучшей модели 

In [ ]:
metrics_train = model_yolov8s_img1024.val(data = "VisDrone.yaml", split = "train", imgsz = 1024)
metrics_val = model_yolov8s_img1024.val(data = "VisDrone.yaml", imgsz = 1024)
metrics_test = model_yolov8s_img1024.val(data = "VisDrone.yaml", split = "test", imgsz = 1024)

In [ ]:
print("yolov8s_img1024_100epochs")
print(f"Train: {metrics_train.box.map}")
print(f"Val: {metrics_val.box.map}")
print(f"Test: {metrics_test.box.map}")